
# <span style="color:rgb(213,80,0)">CONTROLLO AVANZATO E APPLICAZIONI \_\_\_ 11/05/2026 \_\_\_ SALVATORE \_ RAIOLA \_ DE6000008</span>


In [1]:
clear; close all; clc;



## 0) DEFINIZIONI

In [2]:

A = [0,    1,   0,    0;
    -2, -0.6,   1,    0;
     0,    0,   0,    1;
     1,    0,-  3, -0.8];

B = [0,    0;
     1,  0.4;
     0,    0;
     0.3,  1];

C = [1, 0, 0, 0;
     0, 0, 1, 0
     0, 0, 0, 0;
     0, 0, 0, 0];

n = size(A,1);
m = size(B,2);
p = size(C,1);

D = zeros(p,m);



## 1) PROPRIETÀ STRUTTURALI DEL SISTEMA
### 1.1) Stabilità del sistema ciclo aperto

In [3]:

autovalori = eig(A)

autovalori = 4x1 complex
  -0.3713 + 1.8620i
  -0.3713 - 1.8620i
  -0.3287 + 1.1309i
  -0.3287 - 1.1309i

In [4]:

if all(real(autovalori) < 0)
    disp('Il sistema è asintoticamente stabile a ciclo aperto.');
elseif any(real(autovalori) > 0)
    disp('Il sistema è instabile.');
else
    disp('Il sistema è marginalmente stabile.');
end

Il sistema è asintoticamente stabile a ciclo aperto.


### 1.2) Controllabilità della coppia (A,B)

In [5]:

if(rank(ctrb(A,B))==n)
    disp('Il sistema è completamente controllabile');
else
    disp('Il sistema NON è completamente controllabile');
end

Il sistema è completamente controllabile


### 1.3) Osservabilità della coppia (A,C)

In [6]:

if(rank(obsv(A,C))==n)
    disp('Il sistema è completamente osservabile');
else
    disp('Il sistema NON è completamente osservabile');
end

Il sistema è completamente osservabile


## 2) ASSEGNAMENTO AUTOVALORI MEDIANTE METODO DI SYLVESTER
### 2.1) Specifiche

In [7]:
autovalori_desiderati = [-2; -2.5; -3; -3.5];
Lambda = diag(autovalori_desiderati);



### 2.2) Sintesi

In [8]:
% matrice G con elementi casuali
G = rand(2,4);

X = sylvester(A, -Lambda, -B*G);
K_EIG = G/X

K_EIG = 2x4
   -7.9326   -6.6239    6.6460    4.5593
    0.6094    1.5676   -3.9295   -4.9710


### 2.3) Verifica

In [9]:
eig_cl = eig(A+B*K_EIG);

disp('Autovalori desiderati:'); disp(autovalori_desiderati);

Autovalori desiderati:
   -2.0000
   -2.5000
   -3.0000
   -3.5000

In [10]:
disp('Autovalori a ciclo chiuso:'); disp(eig_cl);

Autovalori a ciclo chiuso:
   -3.5000
   -3.0000
   -2.0000
   -2.5000


## 3) CONTROLLO LQR CON AZIONE INTEGRALE E Ts ASSEGNATO
### 3.1) Specifiche

In [11]:
r = [1; -0.5];
x_0_lqr = [1;
           1;
           1;
           1];

Ts = 5;



### 3.2) Sistema Aumentato

In [12]:
C_LQR = [1, 0, 0, 0;
         0, 0, 1, 0];

p_a = size(C_LQR, 1);

A_a = [ A, zeros(n,p_a);
       -C_LQR, zeros(p_a,p_a)]

A_a = 6x6
         0    1.0000         0         0         0         0
   -2.0000   -0.6000    1.0000         0         0         0
         0         0         0    1.0000         0         0
1.0000         0   -3.0000   -0.8000         0         0
   -1.0000         0         0         0         0         0
         0         0   -1.0000         0         0         0

In [13]:
n_a = size(A_a,1);
B_a = [B;
       zeros(p_a,m)]

B_a = 6x2
         0         0
1.0000    0.4000
         0         0
    0.3000    1.0000
         0         0
         0         0


### 3.3) Sintesi

In [14]:
Q_x = eye(n);
Q_z = 10*eye(p_a);
Q_a = blkdiag(Q_x, Q_z);

R_a = eye(m);

% Trasliamo lo spettro di A
alpha = 4/Ts;
A_hat = A_a + alpha*eye(n_a);

for iter = 1:100
    A_hat = A_a + alpha * eye(n_a);

    % Sintesi LQR
    [K_LQR, ~, E] = lqr(A_hat, B_a, Q_a, R_a);

    % Calcolo del polo dominante REALE del ciclo chiuso
    poli_chiusi = eig(A_a - B_a*K_LQR);
    polo_dominante = max(real(poli_chiusi));

    % Verifica se il vincolo Ts della traccia è soddisfatto
    if polo_dominante < -alpha
        fprintf('Sintesi riuscita all''iterazione %d!\n', iter);
        fprintf('Polo dominante: %f (Richiesto < %f)\n', polo_dominante, -alpha);
        break;
    end

    % Se non è soddisfatto, aumenta il peso degli stati (chiedi più velocità)
    % o diminuisci il peso del controllo (permetti più uso dei motori/attuatori)
    Q_a = Q_a * 1.5;
    R_a = R_a * 0.8;
end

Sintesi riuscita all'iterazione 1!
Polo dominante: -1.432487 (Richiesto < -0.800000)

In [15]:
K_LQR

K_LQR = 2x6
   10.3485    5.1635   -1.1619   -1.3499  -11.5243    3.1719
   -0.2394   -0.6787    9.3999    4.6473    2.1925  -12.3233

In [16]:
K_LQR_x = K_LQR(:,1:n)

K_LQR_x = 2x4
   10.3485    5.1635   -1.1619   -1.3499
   -0.2394   -0.6787    9.3999    4.6473

In [17]:
K_LQR_i = K_LQR(:, n+1:end)

K_LQR_i = 2x2
  -11.5243    3.1719
    2.1925  -12.3233


#### Verificare il risultato della simulazione in [VerificaLQR](matlab:open_system('VerificaLQR.slx'))

## 4) COMPRESSIONE DI IMMAGINE SVD


